# w9_20g_5fold.ipynb — 20G 配方五折（swin W=84 × cap 2048）

User (2026-07-22): 把"cap 4096→2048 免费(饱和)+ swin W=84 免费(A4)"这
条钉成五折实测。臂 = **wcle_swin84step42loop2i2ce_icetf @2048**，2000ep，
cvsel 选点(noname h1+h5+2×val_tag，干净探针)，seed=fold。锚点存储只 8.5G（vs 4096 的 17G），单步激活也随
W 与 cap 各减半 → 单塔 ~18G，一张 A100（80G）用 VRAM 调度器打包同时
跑 3–4 折。判据：Stripped hit@1 落在 swin168@4096 冠军（.702±.034）附近、
Name 贴 .94 → "20G 配方几乎不亏分" 成立，进论文一行实测。参照(读出单元从卷上实时读取，随 cvsel 重选刷新，不再硬编码)=
swin168@4096 冠军、i2ce@4096、i2ce@2048；先跑 w9_cv_test_tag.ipynb
把各臂重选到 cvsel 并带上 test_tag，本表方可同口径对比。AUTO-STOPS。


In [ ]:
# constants
import os

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_cv_out"       # CV campaign out dir

RECIPES = ["wcle_swin84step42loop2i2ce_icetf"]   # 20G recipe: W=84 x cap 2048
REF_RECIPES = []                                # custom readout below
CAPS = [2048]
N_FOLDS = 5
EPOCHS, CKPT_EVERY, CKPT_SEEDS, TOPUP_SEEDS = 2000, 50, 2, 10   # ZS: seeds unused

# VRAM scheduler knobs
SAFETY = 0.85
RESERVE_GIB = 1.5

# 48G cards (L40) can't fit the 4096 grad-gallery cells (~45G): pods whose
# smallest GPU has <60GiB free skip caps above MAX_CAP_48G, run their share
# (512/1024/2048 = 30 towers) and AUTO-STOP early -- no waiting on the
# A100 pod, no OOM-burned claims. 80G pods run all 40.
MAX_CAP_48G = 2048

def nm_of(r, k, cap):
    return f"w9cv_{r}_fold{k}" + (f"_g{cap}" if cap != 512 else "")

os.makedirs(OUT_DIR, exist_ok=True)
print("jobs:", len(RECIPES) * N_FOLDS * len(CAPS),
      f"({len(RECIPES)} recipes x {N_FOLDS} folds x {len(CAPS)} caps) @ {EPOCHS}ep")


In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy
        break
import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")


In [ ]:
# Stage the corpus into RAM (same file set as w9_a100.ipynb).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz", "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)

In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# VRAM-BUDGET SCHEDULER over the full (recipe x fold x cap) grid.
# Warmup measures one cost per cap (i2ce fold0, the heavier recipe); the
# ZS-only done marker is tower_<nm>_fp_ep{EPOCHS}.npz (a lower-budget or
# crashed tower auto-continues from its newest ckpt / resume bundle).
import os, subprocess, tempfile, threading, time
from pathlib import Path

cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
# measure runs write to a LOCAL scratch dir: with the real job name on the
# SHARED volume, a measure could load another machine's resume bundle
# (start_ep >= 1 -> zero steps -> no cost file) or race its zs_traj writes.
MEAS_OUT = os.path.join(tempfile.gettempdir(), "w9_measure_out")
os.makedirs(MEAS_OUT, exist_ok=True)
gpus = J.detect_gpus()

def _smi_mib(field, g):
    out = subprocess.check_output(
        ["nvidia-smi", f"--query-gpu={field}", "--format=csv,noheader,nounits",
         "-i", str(g)]).decode().strip().split("\n")[0]
    return int(out) * 2**20

free = {g: _smi_mib("memory.free", g) for g in gpus}
budget = {g: int(free[g] * SAFETY - RESERVE_GIB * 2**30) for g in gpus}
print(f"[vram] budgets {[f'{budget[g] / 2**30:.0f}G' for g in gpus]}")
vram_gib = min(free.values()) / 2**30
CAP_CEIL = MAX_CAP_48G if vram_gib < 60 else 10**9
print(f"[vram] cap ceiling " + (str(CAP_CEIL) if CAP_CEIL < 10**9
                                else "none (runs all caps)"))

todo0 = []
for cap in CAPS:
    for r in RECIPES:
        for k in range(N_FOLDS):
            nm = nm_of(r, k, cap)
            if cap > CAP_CEIL:
                print(f"[skip-vram] {nm} cap {cap} > {CAP_CEIL}"); continue
            if (Path(OUT_DIR) / f"tower_{nm}_fp_ep{EPOCHS}.npz").exists():
                print(f"[skip] {nm} at {EPOCHS}"); continue
            todo0.append((r, k, cap, nm))

cost = {}
for r in sorted({r for r, _k, _c, _n in todo0}):
    # arms differ wildly here (slot8 ~45G, mq/byol/epd far lighter):
    # measure ONE cost per ARM at the (single) cap.
    tf = Path(tempfile.gettempdir()) / f"w9cv_vram_{r}.txt"
    tf.unlink(missing_ok=True)
    cmd = ["python", "-u", J.CV_WORKER, "--data-dir", DATA_DIR, "--out-dir",
           MEAS_OUT, "--repo", REPO, "--arm", r, "--fold", "0",
           "--n-folds", str(N_FOLDS), "--anchor-cap", str(CAPS[0]),
           "--epochs", "1", "--full-pool", "--full-pool-path", FULL_POOL_PATH,
           "--measure-vram", str(tf)]
    print(f"[warmup] {r} ...", flush=True)
    with open(logd / f"measure_cv_{r}.log", "w") as fh:
        subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                       env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpus[0],
                                PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True"))
    cost[r] = int(tf.read_text()) if tf.exists() else budget[gpus[0]] + 1
    if r.startswith("wcle_epdb"):
        # batch=all VICReg's 3-step smoke misses episodic long-review
        # spikes (run-7: 4 folds OOMed when co-packed beside ~40-49G
        # neighbours); floor the cost so the scheduler seats it alone.
        cost[r] = max(cost[r], 48 << 30)
    print(f"[warmup] {r}: {cost[r] / 2**30:.2f}G", flush=True)

todo = sorted(((r, k, cap, nm, cost[r]) for r, k, cap, nm in todo0),
              key=lambda x: -x[4])            # heaviest arm sets the makespan
now_used = {g: 0 for g in gpus}
now_cnt = {g: 0 for g in gpus}   # concurrent workers per GPU
MAX_CO = 4        # host-RAM guard: each worker boot has an ~8.5G transient
LAUNCH_STAGGER = 60
fails = []
retried = set()   # one automatic retry per job (post-mortem: 4 folds died in minute 3, pod idled 22h)
cvn = threading.Condition()

def run_job(g, r, k, cap, nm, c):
    try:
        if not J.try_claim(cdir, nm):
            print(f"[claim] {nm} held elsewhere -- skipped", flush=True); return
        cmd = ["python", "-u", J.CV_WORKER, "--data-dir", DATA_DIR, "--out-dir",
               OUT_DIR, "--repo", REPO, "--arm", r, "--fold", str(k),
               "--n-folds", str(N_FOLDS), "--anchor-cap", str(cap),
               "--epochs", str(EPOCHS), "--ckpt-every", str(CKPT_EVERY),
               "--ckpt-seeds", str(CKPT_SEEDS), "--topup-seeds", str(TOPUP_SEEDS),
               "--full-pool", "--full-pool-path", FULL_POOL_PATH,
               "--claim-file", str(cdir / f"{nm}.claim")]
        t0 = time.time()
        with open(logd / f"{r}_fold{k}_g{cap}.log", "w") as fh:
            p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                               env=dict(os.environ, CUDA_VISIBLE_DEVICES=g,
                                       PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True"))
        if p.returncode != 0:
            (cdir / f"{nm}.claim").unlink(missing_ok=True)
            with cvn:
                if nm not in retried:
                    retried.add(nm)
                    pending.append((r, k, cap, nm, c))
                    print(f"[retry] {nm} re-queued once", flush=True)
                else:
                    fails.append(nm)
        print(f"[gpu{g}] {'ok' if p.returncode == 0 else 'FAIL'} {nm} "
              f"[{(time.time() - t0) / 3600:.1f}h]", flush=True)
    finally:
        with cvn:
            now_used[g] -= c
            now_cnt[g] -= 1
            cvn.notify_all()

stop_evt = threading.Event()
threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True).start()
active = []
with cvn:
    pending = list(todo)
    while pending or any(t.is_alive() for t in active):
        prog = False
        i = 0
        while i < len(pending):
            r, k, cap, nm, c = pending[i]
            fit = [g for g in gpus if now_cnt[g] < MAX_CO
                   and (now_used[g] + c <= budget[g] or now_used[g] == 0)]
            if not fit:
                i += 1; continue
            g = min(fit, key=lambda g: now_used[g])
            now_used[g] += c
            now_cnt[g] += 1
            th = threading.Thread(target=run_job, args=(g, r, k, cap, nm, c),
                                  daemon=True)
            active.append(th); th.start(); pending.pop(i)
            print(f"[sched] {nm} -> gpu{g} ({c / 2**30:.1f}G, used "
                  f"{now_used[g] / 2**30:.1f}/{budget[g] / 2**30:.0f}G)", flush=True)
            prog = True
            cvn.wait(timeout=LAUNCH_STAGGER)   # stagger boots: ~8.5G host-RAM
            # transient per worker load; unstaggered bursts OOM-killed the
            # tau pod kernel (12 towers never launched, 2026-07-20)
        active = [t for t in active if t.is_alive()]
        if not prog:
            cvn.wait(timeout=3)
stop_evt.set()
for t in active:
    t.join()
print(f"FINAL grid drained; {len(fails)} failed")
for nm in fails:
    print("  FAILED:", nm)


In [ ]:
# SELECTION-REPAIR (rerun-safe): towers whose zsbest predates the clean
# selection (no 'val_tag') are re-selected from their saved projections --
# the worker skips training (done-marker), refreshes the traj from the npzs
# (SPq stored per ckpt -> no GPU re-embedding) and rewrites zsbest by cvsel
# = noname h1 + h5 + 2*val_tag (val_tag = clean-inductive tag).
import json as _json, os, queue as _q, subprocess, threading, time
from pathlib import Path

need = []
for cap in CAPS:
    for r in RECIPES:
        for k in range(N_FOLDS):
            nm = nm_of(r, k, cap)
            if not (Path(OUT_DIR) / f"tower_{nm}_fp_ep{EPOCHS}.npz").exists():
                continue
            zb = Path(OUT_DIR) / f"zsbest_{nm}_fp.json"
            if zb.exists() and "val_tag" in _json.loads(zb.read_text()):
                continue
            need.append((r, k, cap, nm))
print(f"{len(need)} tower(s) need re-selection")
jobs = _q.Queue()
for j in need:
    jobs.put(j)
cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"

def rep(g):
    while True:
        try:
            r, k, cap, nm = jobs.get_nowait()
        except _q.Empty:
            return
        if not J.try_claim(cdir, nm):
            print(f"[claim] {nm} held -- skipped", flush=True); continue
        cmd = ["python", "-u", J.CV_WORKER, "--data-dir", DATA_DIR, "--out-dir",
               OUT_DIR, "--repo", REPO, "--arm", r, "--fold", str(k),
               "--n-folds", str(N_FOLDS), "--anchor-cap", str(cap),
               "--epochs", str(EPOCHS), "--full-pool",
               "--full-pool-path", FULL_POOL_PATH,
               "--claim-file", str(cdir / f"{nm}.claim")]
        t0 = time.time()
        with open(logd / f"reselect_{nm}.log", "w") as fh:
            p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                               env=dict(os.environ, CUDA_VISIBLE_DEVICES=g,
                                       PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True"))
        print(f"[gpu{g}] reselect {'ok' if p.returncode == 0 else 'FAIL'} {nm} "
              f"[{(time.time() - t0) / 60:.0f}m]", flush=True)

ths = [threading.Thread(target=rep, args=(g,)) for g in J.detect_gpus()]
for t in ths:
    t.start()
for t in ths:
    t.join()
print("re-selection pass done")


In [ ]:
# 20G-recipe readout: swin84@2048 five-fold vs LIVE references (all read
# from the volume, so every row is whatever selection currently produced --
# after w9_cv_test_tag.ipynb they are all cvsel-selected and carry test_tag).
import json
import numpy as np
from pathlib import Path

def agg(arm, cap):
    rows = []
    for k in range(N_FOLDS):
        p = Path(OUT_DIR) / f"zsbest_{nm_of(arm, k, cap)}_fp.json"
        if p.exists():
            rows.append(json.loads(p.read_text()))
    return rows

def stat(rows, f):
    v = [d[f] for d in rows if f in d]
    return (np.mean(v), np.std(v)) if v else (float("nan"), float("nan"))

REFS = [  # (label, arm, cap, memory)
    ("swin168 @4096 (champ)", "wcle_swin168step84loop2i2ce_icetf", 4096, "17G store"),
    ("i2ce   @4096 (full)",   "wcle_i2ce_icetf",                   4096, "17G store"),
    ("i2ce   @2048 (full)",   "wcle_i2ce_icetf",                   2048, "8.5G store"),
]
print(f"===== 20G recipe: swin84 @ {CAPS[0]} (8.5G store) =====")
print(f"{'arm':30s} {'Name h1':>10} {'Strip h1':>12} {'test_tag':>10}  mem")
champ = None
for lab, arm, cap, mem in REFS:
    rw = agg(arm, cap)
    if not rw:
        print(f"{lab:30s} {'(pending)':>10}"); continue
    n1, s1, tt = stat(rw, "nm_neutral"), stat(rw, "nm_noname"), stat(rw, "test_tag")
    print(f"{lab:30s} {n1[0]:>10.3f} {s1[0]:>9.3f}±{s1[1]:.3f} {tt[0]:>10.3f}   {mem}")
    if arm.startswith("wcle_swin168"):
        champ = s1[0]

rw = agg(RECIPES[0], CAPS[0])
if rw:
    n1, s1, tt = stat(rw, "nm_neutral"), stat(rw, "nm_noname"), stat(rw, "test_tag")
    print(f"{'swin84 @2048 (THIS, 20G)':30s} {n1[0]:>10.3f} "
          f"{s1[0]:>9.3f}±{s1[1]:.3f} {tt[0]:>10.3f}   8.5G store  "
          f"[{len(rw)}/{N_FOLDS} folds]")
    if champ == champ:   # not nan
        dv = s1[0] - champ
        print(f"\n  Stripped h1 vs swin168@4096 champion: {dv:+.3f} "
              f"({'within fold noise -> 20G recipe ~free' if abs(dv) <= 0.035 else 'measurable cost'})")
    print(f"  per-fold Stripped h1: {[round(d['nm_noname'], 3) for d in rw]}")
    if any("test_tag" not in d for d in rw):
        print("  NOTE: some rows lack test_tag -- run w9_cv_test_tag.ipynb "
              "first to re-select all arms by cvsel and backfill test_tag.")
else:
    print(f"{'swin84 @2048 (THIS)':30s}  (pending -- no folds yet)")


In [ ]:
# AUTO-STOP: stop THIS pod when the queue has finished (results live on the
# network volume; idle GPU time is pure waste). Uses the hardened ladder in
# VICReg_review/pod_selfstop.py. Set AUTO_STOP=False to keep the pod alive.
AUTO_STOP = True
if AUTO_STOP:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from VICReg_review import pod_selfstop
    if fails:
        print(f"NOTE: {len(fails)} job(s) FAILED -- logs in {OUT_DIR}/logs; "
              "stopping anyway to avoid idle burn.")
    pod_id, api_key, ctl = pod_selfstop.preflight("")
    pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    print("AUTO_STOP disabled -- remember to stop the pod yourself.")